In [ ]:
import numpy as np

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator


def make_A(theta):
    qc = QuantumCircuit(1)
    qc.ry(2 * theta, 0)
    return qc.to_gate(label="A")

def make_Q(theta):

    A = make_A(theta)

    qc = QuantumCircuit(1)

    qc.append(A, [0])
    qc.z(0)
    qc.append(A.inverse(), [0])
    qc.z(0)

    return qc.to_gate(label="Q")


def make_mlae_circuit(theta, m):

    A = make_A(theta)
    Q = make_Q(theta)

    qc = QuantumCircuit(1, 1)

    # Initial state preparation
    qc.append(A, [0])

    # Apply Q^m
    for _ in range(m):
        qc.append(Q, [0])

    # Measure good state |1>
    qc.measure(0, 0)

    return qc


def run_depth(theta, m, shots=1000):

    simulator = AerSimulator()

    qc = make_mlae_circuit(theta, m)
    qc = transpile(qc, simulator)

    result = simulator.run(qc, shots=shots).result()

    counts = result.get_counts()

    successes = counts.get("1", 0)

    return successes

def log_likelihood(theta_candidate, depths, successes, shots):

    ll = 0

    for m, h in zip(depths, successes):

        p = np.sin((2*m + 1) * theta_candidate) ** 2

        # avoid log(0)
        p = np.clip(p, 1e-12, 1 - 1e-12)

        ll += h * np.log(p)
        ll += (shots - h) * np.log(1 - p)

    return ll


def mlae(true_a, depths, shots=1000):

    # Convert amplitude into angle
    theta_true = np.arcsin(np.sqrt(true_a))

    successes = []

    for m in depths:

        h = run_depth(theta_true, m, shots)

        successes.append(h)

        print(
            f"m={m:2d} | "
            f"successes={h:4d}/{shots} | "
            f"frequency={h/shots:.4f}"
        )

    # Search possible theta values
    theta_grid = np.linspace(
        1e-5,
        np.pi/2 - 1e-5,
        100_000
    )

    likelihoods = np.array([
        log_likelihood(
            theta,
            depths,
            successes,
            shots
        )
        for theta in theta_grid
    ])

    theta_hat = theta_grid[np.argmax(likelihoods)]

    # Convert back to amplitude
    a_hat = np.sin(theta_hat) ** 2

    return a_hat





m= 0 | successes= 286/1000 | frequency=0.2860
m= 1 | successes= 980/1000 | frequency=0.9800
m= 2 | successes=  62/1000 | frequency=0.0620
m= 4 | successes= 752/1000 | frequency=0.7520
m= 8 | successes= 171/1000 | frequency=0.1710

True amplitude:      0.3
MLAE estimate:       0.2999499564280358
Absolute error:      5.004357196419784e-05


In [ ]:
true_a = 0.30

depths = [0, 1, 2, 4, 8]

a_hat = mlae(
    true_a=true_a,
    depths=depths,
    shots=1000
)

print()
print("True amplitude:     ", true_a)
print("MLAE estimate:      ", a_hat)
print("Absolute error:     ", abs(a_hat - true_a))